[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_4_Data_Cleaning.ipynb)

# 06.4: Data Cleaning

Before you can trust any analysis, you need to ask: is the data actually correct? Real datasets have problems — missing values, wrong types, inconsistent strings, duplicate rows. Each of these will silently corrupt your results if you don't address it. This notebook covers the most common cleaning operations: detecting and handling missing values, fixing data types, removing duplicates, and standardizing string columns.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
raw = pd.read_csv(url)
raw.columns = raw.columns.str.lower()
raw = raw.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(raw.shape)
raw.head()

(887, 8)


,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


## Why This Step Exists

You've loaded the data, inspected it, and learned to filter it. Before you can trust any analysis, though, you need to ask: *is the data actually correct?*

Real datasets have problems. Missing values — a passenger's age wasn't recorded. Wrong types — a numeric column loaded as text. Inconsistent strings — `"Male"`, `"male"`, and `"MALE"` all meaning the same thing. Duplicate rows — the same record entered twice. Each of these problems will silently corrupt your analysis if you don't address it.

Data cleaning is unglamorous, but skipping it is the most common source of wrong answers in data analysis. A rough industry estimate is that analysts spend 60–80% of their time here.

## 1. Checking for Missing Values

The first question on any new dataset: *what's missing?*

This Titanic source is already clean — no missing values. That's unusual. Many other versions of this dataset have about 20% of ages missing (not all ships recorded every passenger's age). To make the cleaning tools meaningful, we'll introduce realistic gaps.

In [2]:
# This version is complete — but let's verify
raw.isnull().sum()

survived    0
pclass      0
name        0
sex         0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

In [3]:
# Introduce ~20% missing ages, as seen in other Titanic sources
rng = np.random.default_rng(42)
df = raw.copy()
missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
df.loc[missing_idx, "age"] = np.nan

# Now check
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)
pd.DataFrame({"missing": missing_counts, "pct": missing_pct})

,missing,pct
survived,0,0.0
pclass,0,0.0
name,0,0.0
sex,0,0.0
age,177,20.0
sibsp,0,0.0
parch,0,0.0
fare,0,0.0


In [4]:
# Which rows are affected? Use any(axis=1) — "does any column in this row have a NaN?"
rows_with_missing = df[df.isnull().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)}")
rows_with_missing[["name", "age", "survived"]].head()

Rows with at least one missing value: 177


,name,age,survived
5,Mr. James Moran,NaN,0
19,Mrs. Fatima Masselmani,NaN,1
26,Mr. Farred Chehab Emir,NaN,0
29,Mr. Lalio Todoroff,NaN,0
30,Don. Manuel E Uruchurtu,NaN,0


## 2. Deciding What to Do About Missing Data

Once you know what's missing, you have two choices: **drop** the rows with missing values, or **fill** them with something reasonable. Which you choose depends on two things:

1. *How much data would you lose?* Dropping 20% of rows is significant and may bias your results.
2. *Why is the data missing?* If it's random (some ages just weren't recorded), filling with the median is reasonable. If it's systematic (e.g., the poorest passengers were never asked), filling with a blanket value can introduce bias.

### Dropping with `.dropna()`

In [5]:
df_dropped = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropping any row with missing values: {len(df_dropped)} rows")
print(f"Lost: {len(df) - len(df_dropped)} rows ({(len(df)-len(df_dropped))/len(df):.0%})")

Original: 887 rows
After dropping any row with missing values: 710 rows
Lost: 177 rows (20%)


In [6]:
# More surgical: drop only rows where 'age' is missing
# Other columns (if they had missing values) would still be kept
df_dropped_age = df.dropna(subset=["age"])
print(f"Rows after dropping only age-missing rows: {len(df_dropped_age)}")

Rows after dropping only age-missing rows: 710


### Filling with `.fillna()`

Often you'd rather fill than lose data. The most common approach for a numeric column like age is to fill with the **median** — the middle value — which is more robust to outliers than the mean.

In [7]:
median_age = df["age"].median()
print(f"Median age (computed on non-missing rows): {median_age}")

df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(median_age)
print(f"Missing after fill: {df_filled['age'].isnull().sum()}")

Median age (computed on non-missing rows): 28.0
Missing after fill: 0


In [8]:
# You can fill multiple columns at once with a dict — each column gets its own value
df_filled2 = df.fillna({
    "age": df["age"].median(),    # numeric: use median
    # add more columns here if they had missing values
})
df_filled2.isnull().sum()

survived    0
pclass      0
name        0
sex         0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

> **Machine learning warning:** When you later split data into training and test sets, always compute the fill value (median, mean, mode) from the *training set only*, then apply that same value to the test set. Computing from the full dataset lets information from the test set leak into your model — a subtle but significant bug.

## 3. Duplicate Rows

Duplicates are less common than missing values but worth checking. They can appear when data from multiple sources is combined, or when a data entry process allows re-submission.

In [9]:
print("Duplicates in the original dataset:", raw.duplicated().sum())

Duplicates in the original dataset: 0


In [10]:
# To demonstrate the tools, manufacture some duplicates
df_dupes = pd.concat([raw.head(5), raw.head(3)], ignore_index=True)
print(f"Rows including duplicates: {len(df_dupes)}")
print(f"Duplicate rows detected:   {df_dupes.duplicated().sum()}")

# Show which rows are duplicates
df_dupes[df_dupes.duplicated(keep=False)][["name", "age", "pclass"]]

Rows including duplicates: 8
Duplicate rows detected:   3


,name,age,pclass
0,Mr. Owen Harris Braund,22.0,3
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,38.0,1
2,Miss. Laina Heikkinen,26.0,3
5,Mr. Owen Harris Braund,22.0,3
6,Mrs. John Bradley (Florence Briggs Thayer) Cum...,38.0,1
7,Miss. Laina Heikkinen,26.0,3


In [11]:
df_clean = df_dupes.drop_duplicates()
print(f"Rows after removing duplicates: {len(df_clean)}")

Rows after removing duplicates: 5


## 4. Renaming Columns

Column names from raw data are often inconsistent, long, or hard to type. Rename them early so the rest of your code is cleaner.

In [12]:
# .rename() takes a dict: {old_name: new_name}
df_renamed = raw.rename(columns={
    "sibsp": "siblings_spouses",
    "parch": "parents_children",
    "pclass": "passenger_class",
})
print(df_renamed.columns.tolist())

['survived', 'passenger_class', 'name', 'sex', 'age', 'siblings_spouses', 'parents_children', 'fare']


## 5. Fixing Data Types

Some columns have the wrong type. `survived` is 0 or 1, but it's a *category*, not a number — you'd never add two survived-values together or take their average in a meaningful way. `pclass` is the same. Telling pandas these are categories makes operations on them more reliable and the code more self-documenting.

In [13]:
print("Before:")
print(raw.dtypes)
print()
df_typed = raw.copy()
df_typed["survived"] = df_typed["survived"].astype("category")
df_typed["pclass"]   = df_typed["pclass"].astype("category")
df_typed["sex"]      = df_typed["sex"].astype("category")

print("After:")
print(df_typed.dtypes)

Before:
survived      int64
pclass        int64
name            str
sex             str
age         float64
sibsp         int64
parch         int64
fare        float64
dtype: object

After:
survived    category
pclass      category
name             str
sex         category
age          float64
sibsp          int64
parch          int64
fare         float64
dtype: object


In [14]:
# pd.to_numeric() handles columns that should be numeric but contain bad strings
# errors='coerce' converts unparseable values to NaN rather than crashing
messy_ages = pd.Series(["22", "38", "unknown", "35", "28"])
pd.to_numeric(messy_ages, errors="coerce")

0    22.0
1    38.0
2     NaN
3    35.0
4    28.0
dtype: float64

## 6. Cleaning String Columns

The `name` column is a gold mine of hidden information — each name includes a title (Mr., Mrs., Miss., Master., Dr.) that encodes both sex and approximate age. But to use it, you first need to extract it cleanly.

In [15]:
# Names look like: "Mr. Owen Harris Braund"
# Extract the first word ending in a period
titles = raw["name"].str.extract(r'^(\w+\.)')
titles.value_counts()

0        
Mr.          513
Miss.        182
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Major.         2
Mlle.          2
Col.           2
Don.           1
Mme.           1
Ms.            1
Lady.          1
Sir.           1
Capt.          1
Jonkheer.      1
Name: count, dtype: int64

Those four titles cover almost everyone. `Master.` was the formal title for boys under roughly 14, so it's actually a proxy for age. `Miss.` was used for unmarried women of any age.

Fixing inconsistencies in string data is also common. Imagine the `sex` column had been entered by hand:

In [16]:
# A messy sex column from manual data entry
messy = pd.Series(["  Male", "female", "MALE", "Female ", "male"])
cleaned = messy.str.strip().str.lower()
print(cleaned.unique())

<StringArray>
['male', 'female']
Length: 2, dtype: str


## 7. A Complete Cleaning Workflow

Here is a realistic end-to-end cleaning function. It takes the raw CSV and produces a tidy, analysis-ready DataFrame.

In [17]:
def prepare_titanic(raw_df, missing_age_seed=42):
    df = raw_df.copy()

    # Standardize column names
    df.columns = df.columns.str.lower()
    df = df.rename(columns={
        'siblings/spouses aboard': 'sibsp',
        'parents/children aboard': 'parch',
    })

    # Introduce realistic missing ages (~20%), then fill with median
    # (Skip this block if your source already has missing ages)
    rng = np.random.default_rng(missing_age_seed)
    missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
    df.loc[missing_idx, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())

    # Extract title from name, then drop name (too noisy to use directly)
    df["title"] = df["name"].str.extract(r'^(\w+\.)')
    df = df.drop(columns=["name"])

    # Derive a family feature
    df["has_family"] = ((df["sibsp"] > 0) | (df["parch"] > 0)).astype(int)

    # Fix data types
    for col in ["survived", "pclass", "sex", "title"]:
        df[col] = df[col].astype("category")

    return df

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
clean = prepare_titanic(pd.read_csv(url))

print(clean.dtypes)
print()
print(clean.isnull().sum())
print()
clean.head()

survived      category
pclass        category
sex           category
age            float64
sibsp            int64
parch            int64
fare           float64
title         category
has_family       int64
dtype: object

survived      0
pclass        0
sex           0
age           0
sibsp         0
parch         0
fare          0
title         1
has_family    0
dtype: int64



,survived,pclass,sex,age,sibsp,parch,fare,title,has_family
0,0,3,male,22.0,1,0,7.2500,Mr.,1
1,1,1,female,38.0,1,0,71.2833,Mrs.,1
2,1,3,female,26.0,0,0,7.9250,Miss.,0
3,1,1,female,35.0,1,0,53.1000,Mrs.,1
4,0,3,male,35.0,0,0,8.0500,Mr.,0


## What's next

You have a cleaned, trustworthy dataset. The next question is comparison: how do different groups of passengers differ from one another? Computing the overall average fare tells you one number about 887 passengers. Computing the average fare separately for each class tells you something far more interesting. Notebook 06.5 introduces GroupBy — the operation that divides the data into groups, computes a summary within each group, and combines the results into a table you can compare.